`.groupby()`  It splits your data into groups, then applies an aggregate method to each group separately.  

`df.groupby('department')['id'].count()`

---

`.reset_index()` — Why You Need It   
After `.groupby().count()`, **the group column (department) becomes the DataFrame's index** instead of a regular column. The index is like a row label — useful internally, but awkward for further analysis.  
`.reset_index()` moves it back to a normal column. The name= parameter renames the aggregated column: `.reset_index(name="employee_count")` turns the unnamed count into a descriptive column name. You'll chain .reset_index() after almost every groupby.

### Η Λειτουργία του `reset_index()`

Η εντολή `reset_index()` χρησιμοποιείται για να επαναφέρει το ευρετήριο (index) ενός πίνακα στην αρχική του κατάσταση (αριθμητικός δείκτης 0, 1, 2...) και να μετατρέψει τα υπάρχοντα ευρετήρια σε κανονικές στήλες.

---

#### 🔥 Τι κάνει η εντολή στο παράδειγμά μας;

```python
orders.groupby("cust_id")["total_order_cost"].sum().reset_index(name="total_spent")
```

Όταν κάνουμε `groupby("cust_id")`, η στήλη `cust_id` γίνεται το **Index (κλειδί)** του αποτελέσματος. 
Το `reset_index(name="total_spent")` κάνει δύο πράγματα:
1. **Ξεκλειδώνει το `cust_id`** και το μετατρέπει ξανά σε κανονική, επεξεργάσιμη στήλη.
2. **Μετατρέπει το αποτέλεσμα σε DataFrame** (πίνακα) και ονομάζει τη στήλη του αθροίσματος ως `total_spent`.

---

#### ❌ Τι θα γινόταν ΧΩΡΙΣ το `reset_index()`

Αν εκτελούσατε μόνο αυτό:
```python
orders.groupby("cust_id")["total_order_cost"].sum()
```

1. **Αλλαγή Δομής (Series αντί για DataFrame):** Το αποτέλεσμα θα ήταν μια απλή σειρά (Pandas Series) και όχι ένας πλήρης πίνακας με στήλες.
2. **Κλειδωμένο Index:** Το `cust_id` θα ήταν index. Αυτό σημαίνει ότι δεν θα μπορούσατε να το χρησιμοποιήσετε εύκολα για άλλες πράξεις (π.χ. εξαγωγή σε Excel ή Merge με άλλον πίνακα).
3. **Λάθος Όνομα Στήλης:** Η στήλη των ποσών θα συνεχιζόταν να λέγεται `total_order_cost`, παρόλο που πλέον περιέχει το *άθροισμα* και όχι το κόστος μιας μεμονωμένης παραγγελίας.

---

#### 📊 Οπτική Σύγκριση Αποτελεσμάτων

#### 1. Χωρίς το `reset_index()` (Δομή: Series)
Το `cust_id` είναι index (διακριτό οπτικά) και η στήλη διατηρεί το παλιό όνομα.

```text
cust_id
101       150.50
102        45.00
Name: total_order_cost, dtype: float64
```

#### 2. Με το `reset_index(name="total_spent")` (Δομή: DataFrame)
Ένας καθαρός πίνακας με αυτόματη αρίθμηση (0, 1...) αριστερά και δύο κανονικές στήλες.

```text
   cust_id  total_spent
0      101       150.50
1      102        45.00
```


In [1]:
import pandas as pd

In [2]:
movies= pd.read_csv("../data/movies.csv", index_col=0)

In [3]:
movies.columns

Index(['year', 'genre', 'runtime_min', 'budget_musd', 'revenue_musd', 'rating',
       'votes'],
      dtype='str')

In [4]:
# χωρίς reset_index
movies.groupby('genre')['genre'].count()

genre
Action         79
Animation      70
Comedy         86
Documentary    87
Drama          88
Thriller       90
Name: genre, dtype: int64

In [5]:
# με reset_index
movies.groupby('genre')['genre'].count().reset_index(name='count')

,genre,count
0,Action,79
1,Animation,70
2,Comedy,86
3,Documentary,87
4,Drama,88
5,Thriller,90


In [6]:
movies.groupby('genre')['revenue_musd'].agg(movies_count='count',avg_revenue='mean').reset_index()

,genre,movies_count,avg_revenue
0,Action,79,104.705063
1,Animation,70,89.394286
2,Comedy,86,73.906977
3,Documentary,87,72.132184
4,Drama,88,98.428409
5,Thriller,90,82.548889


In [7]:
(
    movies.groupby('genre')
    .agg(movie_count=('genre','count'),
         avg_budget=('budget_musd','mean'),
         avg_revenue=('revenue_musd','mean')
    ).round(2)
    .reset_index()
    .sort_values('avg_revenue', ascending=False)         
)

,genre,movie_count,avg_budget,avg_revenue
0,Action,79,49.13,104.71
4,Drama,88,45.93,98.43
1,Animation,70,44.20,89.39
5,Thriller,90,39.53,82.55
2,Comedy,86,43.77,73.91
3,Documentary,87,39.99,72.13


• **Όχι `reset_index`**: Όταν είστε στο Jupyter Notebook / Terminal και απλώς κοιτάτε, διαβάζετε ή κάνετε ένα γρήγορο γράφημα.  
• **Ναι `reset_index`**: Όταν το αποτέλεσμα πρόκειται να τροφοδοτήσει άλλον κώδικα, άλλη συνάρτηση ή να εξαχθεί εκτός Python.


In [8]:
# είναι Series δεν χρειάζεται όρισμα ονόματος στο sort_values
movies.groupby('genre')['genre'].count().sort_values(ascending=False)

genre
Thriller       90
Drama          88
Documentary    87
Comedy         86
Action         79
Animation      70
Name: genre, dtype: int64

In [9]:
# έβαλα reset_index είναι DataFrame, άρα στο sort_values χρειάζεται όρισμα ονόματος 
movies.groupby('genre')['genre'].count().reset_index(name='count').sort_values('count', ascending=False)

,genre,count
5,Thriller,90
4,Drama,88
3,Documentary,87
2,Comedy,86
0,Action,79
1,Animation,70


In [10]:
movies.groupby('genre')['revenue_musd'].mean().round(2).reset_index(name='avg_revenue').sort_values('avg_revenue', ascending=False)

,genre,avg_revenue
0,Action,104.71
4,Drama,98.43
1,Animation,89.39
5,Thriller,82.55
2,Comedy,73.91
3,Documentary,72.13


#### Method Chaining
όταν ένας κώδικας είναι μακρύς, μπορώ να τον κλείσω μέσα σε παρενθέσεις και να σπάω σε νέα γραμμή τον κώδικα πριν από κάθε τελεία. Μπορώ να προσθέτω και σχόλια 

In [11]:
(
    movies
    .groupby('genre')['revenue_musd']  # 1. Ομαδοποίηση ανά είδος και επιλογή εσόδων
    .mean()                            # 2. Υπολογισμός μέσου όρου
    .round(2)                          # 3. Στρογγυλοποίηση σε 2 δεκαδικά
    .reset_index(name='avg_revenue')   # 4. Μετατροπή σε DataFrame με νέο όνομα
    .sort_values('avg_revenue', ascending=False) # 5. Φθίνουσα ταξινόμηση
)


,genre,avg_revenue
0,Action,104.71
4,Drama,98.43
1,Animation,89.39
5,Thriller,82.55
2,Comedy,73.91
3,Documentary,72.13


In [23]:
(
    movies.loc[
        movies['genre'].isin(['Action', 'Thriller', 'Comedy'])     
    ]
    .groupby('genre')['genre']
    .count()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

,genre,count
2,Thriller,90
1,Comedy,86
0,Action,79


In [43]:
movies.groupby('genre').size().reset_index(name='count')

,genre,count
0,Action,79
1,Animation,70
2,Comedy,86
3,Documentary,87
4,Drama,88
5,Thriller,90


In [ ]:
movies['genre'].value_counts().reset_index()

,genre,count
0,Thriller,90
1,Drama,88
2,Documentary,87
3,Comedy,86
4,Action,79
5,Animation,70


In [42]:
(movies['genre'].value_counts(normalize=True)*100).reset_index(name='pct %')

,genre,pct %
0,Thriller,18.0
1,Drama,17.6
2,Documentary,17.4
3,Comedy,17.2
4,Action,15.8
5,Animation,14.0


In [13]:
# με την aggregate δημιουργούνται αυτόματα τα ονόματα των στηλών
movies.groupby('genre')['revenue_musd'].aggregate(['mean','median']).round(2).reset_index()

,genre,mean,median
0,Action,104.71,74.80
1,Animation,89.39,54.95
2,Comedy,73.91,56.60
3,Documentary,72.13,48.30
4,Drama,98.43,54.00
5,Thriller,82.55,58.55


In [14]:
# αν θέλω να μετονομάσω τις στήλες των aggregate functions --> μέσα στην aggregate
(
    movies.groupby('genre')['revenue_musd']
    .aggregate(avg_revenue='mean',median_revenue='median',skewness='skew')
    .round(2)
    .reset_index()
    .sort_values('avg_revenue', ascending=False)
)

,genre,avg_revenue,median_revenue,skewness
0,Action,104.71,74.80,1.08
4,Drama,98.43,54.00,1.81
1,Animation,89.39,54.95,2.45
5,Thriller,82.55,58.55,1.90
2,Comedy,73.91,56.60,1.21
3,Documentary,72.13,48.30,1.76


In [15]:
#group by 2 columns
(
    movies[movies['year']==2025]
    .groupby(['year','genre'])['genre']
    .count()
    .reset_index(name='count')
    .sort_values(['year','count'],ascending=[True,False])
)

,year,genre,count
3,2025,Documentary,6
4,2025,Drama,5
5,2025,Thriller,5
0,2025,Action,3
2,2025,Comedy,3
1,2025,Animation,2


In [16]:
# την τελευταία δεκαετία, έχουν βγει περισσότερες ταινίες Drama ή Action, και ποιες έιχαν το 
# περισσότερο μέσο και διάμεσο revenue 
(
    movies[
        (movies['year'].between(2015,2025))&
        (movies['genre'].isin(['Action','Drama']))
    ]
    .groupby('genre')['revenue_musd']
    .aggregate(average_revenue='mean', median_revenue='median',movies_count='count')
    .round(2)
    .sort_values(by=['average_revenue'], ascending=[False])
    .reset_index()
)

,genre,average_revenue,median_revenue,movies_count
0,Action,141.52,127.35,30
1,Drama,116.92,68.65,34


#### Για να κάνετε συνάθροιση σε διαφορετικές στήλες με διαφορετικές συναρτήσεις, θα χρησιμοποιήσετε πάλι την `.aggregate()`, αλλά αυτή τη φορά θα της **περάσετε ένα Python Dictionary** (λεξικό).

In [17]:
(
    movies.groupby('genre')
    .agg({'revenue_musd':['count','mean','median'],'budget_musd':['mean','median']})
    .reset_index()
    .sort_values(('revenue_musd','mean'), ascending=False)
)

genre revenue_musd                    budget_musd       
                      count        mean median        mean median
0       Action           79  104.705063  74.80   49.132911  42.80
4        Drama           88   98.428409  54.00   45.929545  43.60
1    Animation           70   89.394286  54.95   44.195714  37.25
5     Thriller           90   82.548889  58.55   39.531111  37.60
2       Comedy           86   73.906977  56.60   43.770930  40.65
3  Documentary           87   72.132184  48.30   39.990805  34.30

In [18]:
# δευτερος τρόπος αν θέλω να αλλάξω ονόματα των συναθροίσεων (μέσα στο agg)
(
    movies.groupby('genre')
    .agg(movies_count=('genre','count'),
         avg_revenue=('revenue_musd','mean'),
         avg_budget=('budget_musd','mean')
         ).round(2)
         .sort_values('avg_revenue', ascending=False)
         .reset_index()
)

,genre,movies_count,avg_revenue,avg_budget
0,Action,79,104.71,49.13
1,Drama,88,98.43,45.93
2,Animation,70,89.39,44.20
3,Thriller,90,82.55,39.53
4,Comedy,86,73.91,43.77
5,Documentary,87,72.13,39.99


In [19]:
movies.groupby(['year','genre'])['genre'].count().reset_index(name='movies_count').sort_values(by=['year','genre'],ascending=[True,True])

,year,genre,movies_count
0,1998,Action,8
1,1998,Animation,2
2,1998,Comedy,4
3,1998,Documentary,2
4,1998,Drama,2
...,...,...,...
151,2025,Animation,2
152,2025,Comedy,3
153,2025,Documentary,6
154,2025,Drama,5


In [20]:
movies.groupby(['year','genre']).agg(movies_count=('genre','count')).reset_index().sort_values(by=['year','genre'],ascending=[True,True])

,year,genre,movies_count
0,1998,Action,8
1,1998,Animation,2
2,1998,Comedy,4
3,1998,Documentary,2
4,1998,Drama,2
...,...,...,...
151,2025,Animation,2
152,2025,Comedy,3
153,2025,Documentary,6
154,2025,Drama,5


In [21]:
movies.groupby('year').agg(count_per_year=('genre','count')).reset_index()

,year,count_per_year
0,1998,23
1,1999,13
2,2000,20
3,2001,14
4,2002,15
5,2003,18
6,2004,19
7,2005,22
8,2006,19
9,2007,25
